In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
 
warnings.filterwarnings("ignore")
np.random.seed(42)
 
os.makedirs("../models", exist_ok=True)

In [ ]:
# STEP 1 — LOAD DATA
# =============================================================================
 
df = pd.read_csv("../data/raw/all_crypto_currencies.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
 
print("=" * 60)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range   : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique coins : {df['slug'].nunique()}")

In [ ]:
# STEP 2 — DATA QUALITY CHECKS
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 2 — DATA QUALITY")
 
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print(f"\nMissing values  : {len(missing)} columns affected" if len(missing) else "\nMissing values  : none")
 
# Duplicates
full_dupes = df.duplicated().sum()
key_dupes  = df.duplicated(subset=['date', 'slug']).sum()
print(f"Full duplicates : {full_dupes}")
print(f"Key duplicates  : {key_dupes}  (date + slug)")
 
# Empty strings
empty = (df == "").sum()
empty = empty[empty > 0]
print(f"Empty strings   : {len(empty)} columns affected" if len(empty) else "Empty strings   : none")
 
# Note: symbol collisions exist (e.g. BITS = bitswift + bitstar)
# These are different coins sharing a ticker — slug is the unique identifier
symbol_collisions = df.groupby('symbol')['slug'].nunique()
print(f"Symbol collisions (same ticker, diff slug): "
      f"{(symbol_collisions > 1).sum()} symbols")

In [ ]:
# STEP 3 — FEATURE ENGINEERING
# (on full df before split — all features are backward-looking per coin,
#  so no future data leaks into earlier rows)
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 3 — FEATURE ENGINEERING")
 
df = df.sort_values(['slug', 'date']).reset_index(drop=True)
grouped = df.groupby('slug', group_keys=False)
 
# --- Returns ---
df['daily_return'] = grouped['close'].pct_change()
 
# --- Moving averages ---
df['ma_30'] = (grouped['close']
               .rolling(30, min_periods=1).mean()
               .reset_index(level=0, drop=True))
 
# --- Volatility ---
df['vol_7']  = (grouped['daily_return']
                .rolling(7,  min_periods=1).std()
                .reset_index(level=0, drop=True))
df['vol_30'] = (grouped['daily_return']
                .rolling(30, min_periods=1).std()
                .reset_index(level=0, drop=True))
 
# --- Rolling range ---
df['rolling_max_7'] = (grouped['high']
                       .rolling(7, min_periods=1).max()
                       .reset_index(level=0, drop=True))
df['rolling_min_7'] = (grouped['low']
                       .rolling(7, min_periods=1).min()
                       .reset_index(level=0, drop=True))
 
# --- Lag features ---
df['lag_1'] = grouped['close'].shift(1)
df['lag_7'] = grouped['close'].shift(7)
 
# --- Momentum ---
df['momentum_7']  = df['close'] - df['lag_7']
df['momentum_14'] = df['close'] - grouped['close'].shift(14)
 
# --- Volume ---
vol_ma_7       = (grouped['volume']
                  .rolling(7, min_periods=1).mean()
                  .reset_index(level=0, drop=True))
df['vol_ratio'] = df['volume'] / (vol_ma_7 + 1e-9)
 
# --- Price spreads ---
df['high_low_spread']    = df['high'] - df['low']
df['close_open_spread']  = df['close'] - df['open']
df['high_close_ratio']   = df['high'] / (df['close'] + 1e-9)
 
# --- Market context ---
df['market_cap_ratio'] = (df['market']
                          / df.groupby('date')['market'].transform('sum'))
df['rank_normalized']  = df['ranknow'] / df['ranknow'].max()
 
# --- EMA ---
df['ema_14'] = grouped['close'].transform(
    lambda x: x.ewm(span=14, adjust=False).mean())
 
# --- RSI (14) ---
def compute_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.rolling(period, min_periods=1).mean()
    avg_loss = loss.rolling(period, min_periods=1).mean()
    rs       = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))
 
df['rsi_14'] = grouped['close'].transform(compute_rsi)
 
# --- MACD ---
ema_12       = grouped['close'].transform(lambda x: x.ewm(span=12, adjust=False).mean())
ema_26       = grouped['close'].transform(lambda x: x.ewm(span=26, adjust=False).mean())
df['macd']   = ema_12 - ema_26
df['macd_signal'] = (grouped['macd']
                     .transform(lambda x: x.ewm(span=9, adjust=False).mean()))
 
# --- Volatility clustering ---
df['vol_cluster_14'] = (grouped['daily_return']
                        .rolling(14, min_periods=1).std()
                        .reset_index(level=0, drop=True))
 
# Fill NaNs introduced by rolling/shift (early rows of each coin)
df.fillna(0, inplace=True)
 
print(f"Feature engineering complete: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
# STEP 4 — DEFINE TARGET
# Next-day closing price per coin — defined BEFORE split so dropna
# removes the last row of each coin cleanly before proportions are calculated
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 4 — TARGET DEFINITION")
 
df['target'] = df.groupby('slug')['close'].shift(-1)
df = df.dropna(subset=['target']).reset_index(drop=True)
 
print(f"Rows after dropping last-row NaNs: {df.shape[0]:,}")

In [ ]:
# STEP 5 — CHRONOLOGICAL TRAIN / VAL / TEST SPLIT  (70 / 15 / 15)
# Split by date so no future data appears in training
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 5 — CHRONOLOGICAL SPLIT")
 
df_sorted   = df.sort_values('date').reset_index(drop=True)
total       = len(df_sorted)
date_counts = df_sorted.groupby('date').size().reset_index(name='count')
date_counts['cum'] = date_counts['count'].cumsum()
date_counts['pct'] = date_counts['cum'] / total
 
train_end = date_counts.loc[date_counts['pct'] >= 0.70, 'date'].iloc[0]
val_end   = date_counts.loc[date_counts['pct'] >= 0.85, 'date'].iloc[0]
 
train_df = df_sorted[df_sorted['date'] <= train_end].copy()
val_df   = df_sorted[(df_sorted['date'] > train_end) & (df_sorted['date'] <= val_end)].copy()
test_df  = df_sorted[df_sorted['date'] > val_end].copy()
 
print(f"Train : {len(train_df):>8,} rows  (up to {train_end.date()})")
print(f"Val   : {len(val_df):>8,} rows  (up to {val_end.date()})")
print(f"Test  : {len(test_df):>8,} rows  (after {val_end.date()})")

In [ ]:
# STEP 6 — FEATURE SELECTION
# Exclude raw price levels (open/high/low/close/market) — the model predicts
# tomorrow's close from *derived* signals; lag_1 captures price persistence
# without exposing today's raw price directly.
# =============================================================================
 
EXCLUDE  = {'slug', 'symbol', 'name', 'date',
            'open', 'high', 'low', 'close', 'market',
            'target'}
FEATURES = [c for c in df.columns if c not in EXCLUDE]
 
print("\n" + "=" * 60)
print(f"STEP 6 — FEATURE SELECTION  ({len(FEATURES)} features)")
print(FEATURES)
 
X_train, y_train = train_df[FEATURES], train_df['target']
X_val,   y_val   = val_df[FEATURES],   val_df['target']
X_test,  y_test  = test_df[FEATURES],  test_df['target']

In [ ]:
# STEP 7 — TRAIN RANDOM FOREST
# No scaling needed — tree splits are invariant to feature scale
# =============================================================================
 
print("\n" + "=" * 60)
print("STEP 7 — TRAINING")
 
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=10,   # prevents overfitting to low-volume altcoins
    max_features=0.5,      # 50% feature sampling per split — reduces variance
    n_jobs=-1,
    random_state=42,
    verbose=1
)
 
rf.fit(X_train, y_train)
print("Training complete.")

In [ ]:
# =============================================================================
# STEP 8 — EVALUATE (Train + Validation)
# =============================================================================

print("\n" + "=" * 60)
print("STEP 8 — EVALUATION")

def evaluate(model, X, y, label):
    preds = model.predict(X)
    rmse  = np.sqrt(mean_squared_error(y, preds))
    r2    = r2_score(y, preds)
    print(f"\n  {label}")
    print(f"    RMSE : {rmse:.6f}")
    print(f"    R²   : {r2:.4f}")
    return preds

train_preds = evaluate(rf, X_train, y_train, "Train")
val_preds   = evaluate(rf, X_val,   y_val,   "Validation")

# Overfit diagnostic
train_r2 = r2_score(y_train, train_preds)
val_r2   = r2_score(y_val,   val_preds)
gap      = train_r2 - val_r2
print(f"\n  Overfit gap (Train R² - Val R²): {gap:.4f}")
if gap > 0.05:
    print("  ⚠  Gap > 0.05 — consider reducing max_depth or increasing min_samples_leaf")
else:
    print("  ✓  Gap acceptable — model generalises well")


In [ ]:
# =============================================================================
# STEP 9 — FINAL TEST EVALUATION (RUN ONCE)
# =============================================================================

print("\n" + "=" * 60)
print("STEP 9 — FINAL TEST EVALUATION")

test_preds = evaluate(rf, X_test, y_test, "Test")

# Summary table
print("\n  ── Summary ──────────────────────────────")
print(f"  {'Split':<12} {'RMSE':>14}  {'R²':>8}")
print(f"  {'─'*38}")
for split, y_true, preds in [
    ("Train",      y_train, train_preds),
    ("Validation", y_val,   val_preds),
    ("Test",       y_test,  test_preds),
]:
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    r2   = r2_score(y_true, preds)
    print(f"  {split:<12} {rmse:>14.6f}  {r2:>8.4f}")

In [ ]:

import joblib
import os

# Ensure directory exists
os.makedirs("models", exist_ok=True)

# Save the trained model and the feature list
joblib.dump(rf, "models/rf_model.pkl")       # rf is your trained RandomForestRegressor
joblib.dump(FEATURES, "models/features.pkl") # FEATURES from Step 6

print("✅ Model and feature list saved successfully!")